# B2.13 · Grey-box agentic pentest — one credential per role, and the matrix it fills

**Function B — Application Security with an AI SDLC → The AI SDLC — an Agentic AppSec Pipeline, Before and After Deploy**

Builds on **[B2.12 · Black-box agentic pentest — inference, and refusing to report it as fact](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**.

| | |
|---|---|
| Tools used | OpenAPI |

## What this lesson is

**What it covers.** A grey-box engagement: filling a roles-by-objects-by-verbs matrix from one credential per role, flagging design mismatches, and ranking the untested cells by blast radius.

**Why a security engineer needs it.** This is the mode that can actually find broken object-level authorisation, and the one most often reported wrong — because authorisation lives in cells, not endpoints, and touching every endpoint leaves most cells untested.

| | |
|---|---|
| **Day 0 — why** | Object-level authorisation is assumed correct because the endpoint list was covered, and broken object access lives in the cells nobody enumerated. |
| **Day 1 — how** | Fill a roles-by-objects-by-verbs matrix from one credential per role, and mark every cell tested, mismatched or untested. |
| **Day 2 — measure** | Cell coverage, not endpoint coverage: 12 of 30 cells on the sample, with two mismatches and the two highest-cost cells never tested. |

## 1 · The hook

The engagement touched every endpoint and called itself complete. Authorisation does not live in endpoints — it lives in cells, one per role per object per verb — and the two that would have mattered here were never sent a single request.

> **At CyberTravels.** The matrix is CyberTravels' three roles against its objects, and the two highest-cost untested cells are both on the audit log — the records a regulator asks for first, and the ones no request in the engagement ever touched.

## 2 · The framework

```
   authorisation is a CELL, not an endpoint

              booking(own)  booking(other)  refund  audit_log
   traveller     ok            ALLOW(!)      deny      ?
   agent-svc     ok               ?          allow     ?
   finance       ok            allow         allow     allow

   ALLOW(!) = design says deny, estate says allow -> finding
   ?        = never tested. ranked by blast radius, not schema order.

   endpoint coverage: 100%.   cell coverage: 40%.
   the two audit_log cells nobody sent a request to are the ? that matter.
```

Grey box is the mode that can actually find broken object-level authorisation,
because it holds more than one identity. It is also the mode most often reported
wrong, because the engagement touches every endpoint, calls itself complete, and
never notices that authorisation is a property of the **cell** — this role, this
object, this verb — and not of the endpoint.

The matrix is the whole method. Roles down one axis, objects and verbs across
the other, and every cell in one of three states: tested, with what came back;
mismatched, where the estate disagrees with the design; or **untested**, which
is data rather than a gap to hide.

Two numbers fall out that an endpoint-coverage report cannot produce. The
mismatches are the findings. The untested cells, ranked by what a wrong answer
would cost rather than by the order the endpoints appear in the schema, are the
backlog — and the coverage fraction is what stops "we tested everything"
standing when 40% of the cells were never sent a request.

One credential per role is the minimum and the enabling fact. With one account
you can test that a feature works; the second account is what turns a
functionality test into a security test.

## 3 · The procedure, as a skill

The skill fills the roles-by-objects-by-verbs matrix for CyberTravels against what the design intends, flags the cells where the estate disagrees, and ranks the untested cells by blast radius rather than by schema order.

### The skill — [`skills/redteam/greybox-authorization-matrix/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/greybox-authorization-matrix/SKILL.md)

```yaml
name: greybox-authorization-matrix
description: >-
  With one credential per role and a schema but no source, fill a
  roles-by-objects-by-verbs matrix, flag design mismatches, and rank the cells
  nobody tested by blast radius. Use for a grey-box engagement, or when
  endpoint coverage is being reported as authorisation coverage.
allowed-tools: Read, Grep, Glob
```

# Authorisation lives in cells, not endpoints

Grey box is the mode that can actually find broken object-level authorisation,
because it has more than one identity. It is also the mode most often reported
wrong: the engagement touches every endpoint, calls itself complete, and never
notices that authorisation is a property of the *cell* — this role, this object,
this verb — not of the endpoint.

The matrix makes the untested cells visible, which is the whole point.

## When to use this

An engagement with low-privilege accounts and a schema, a BOLA/BFLA hunt, and
any report whose coverage claim is a count of endpoints.

## Procedure

**1 — List what you were handed.** One credential per role is the minimum and
the enabling fact; also the schema, a couple of ids you own and a couple you
must not, the matrix as designed, and the scope so the run is not an incident.

**2 — Build the full matrix.** Roles × objects × verbs. Write down what the
design *says* each cell should be, before touching anything.

**3 — Exercise cells and record what came back.** Every cell you do not exercise
stays untested — that is data, not a gap to hide.

**4 — Report three things.** Mismatches, where the estate disagrees with the
design. Untested cells, ranked by what a wrong answer would cost, not by schema
order. And the coverage fraction, so "we tested everything" cannot stand when
40% of the cells were never sent a request.

## Example

```
tested 12 of 30 cells (40%)
2 mismatch(es):
  ! traveller can read booking(other): designed deny, observed allow
```

The run continues past this. The script is the example: `test_skills.py`
executes it on every build, so this block cannot drift from what the skill
actually prints.

## Output contract

```json
{
  "data_sources": [{"source": "str", "gives": "str"}],
  "cells": [{"role": "str", "object": "str", "verb": "str",
             "design": "allow|deny", "observed": "allow|deny|null"}],
  "mismatches": [{"role": "str", "object": "str", "verb": "str",
                  "design": "str", "observed": "str"}],
  "untested": [{"role": "str", "object": "str", "verb": "str", "blast": 0}],
  "coverage": 0.0
}
```

## Failure modes

- **Reporting endpoint coverage as authorisation coverage.** Every endpoint
  touched, most cells untested.
- **Ranking untested cells by schema order.** The audit-log write matters more
  than the profile read; the order they appear in the spec is noise.
- **Testing only your own objects.** One id is a functionality test; the second
  id is the security test.
- **Trusting the design column.** It is what the team believes is enforced,
  which is the thing under test.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/greybox-authorization-matrix/scripts/greybox_authorization_matrix.py
SCRIPT = "skills/redteam/greybox-authorization-matrix/scripts/greybox_authorization_matrix.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.5 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The engagement exercises 12 of 30 cells (40%) and finds two mismatches — a traveller reading another traveller's booking and the agent service writing one — while the two highest-cost cells, both on the audit log, were never sent a request. Endpoint coverage would have called this complete.

## Your turn

Build the matrix for one service you own with two real accounts. The cell you least want to test — an admin verb on another tenant's object — is the one whose result you most need to know.

---

**Next → [B2.14 · Bonus — testing safely: the controls an offensive agent runs inside](https://spbreed.github.io/cyber-commons/lessons/B2.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*